# Part 4 — Feature Pipeline
## Notebook 05. Historical financial features, event labels, and training-only preprocessing

**Research objective.** Construct a 21-feature bank-quarter table from the 48-field validated Financials panel, define a four-quarter failure target using confirmed FDIC `RESTYPE=FAILURE` events, document eligibility and censoring, and establish chronological development and sealed holdout samples.

**Source contract.** The selected experiment must contain verified `acquisition`, `storage`, `quality_panel`, and `processing` stages. This notebook does not download FDIC data or select an input directory using a historical run ID.

**Temporal policy.** The prediction label is defined over `(REPDTE, REPDTE + 4 calendar quarters]`. The 90-day reporting availability delay is an assumption, not a verified historical disclosure date. Reports published after a prospective event under that assumption are not eligible for model development.

**Evaluation boundary.** Part 4 constructs and audits the target, but does not fit a predictive model, tune hyperparameters, select an alarm threshold, or evaluate the test set. Part 5 must refit preprocessing on the training portion of each chronological validation fold. All FDIC source files and published upstream evidence remain read-only.

### 1. Resolve the experiment, restore source revision, and install dependencies

The active experiment pointer selects the verified upstream stages. If the feature stage is published, its recorded source revision is restored for read-only verification. Otherwise, the notebook uses the current committed repository revision, including after an interrupted unpublished build. Pinned packages are installed only when necessary.


In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BFT65B_11230945')
active_file = DRIVE_ROOT / 'control' / 'active_experiment.json'
assert active_file.is_file(), 'Complete Notebooks 01–04 before Part 4'
EXPERIMENT_ID = json.loads(active_file.read_text(encoding='utf-8'))['experiment_id']
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / EXPERIMENT_ID
assert (EXPERIMENT_DIR / 'experiment_config.lock.json').is_file()
REPO_DIR = Path('/content/bft65b_code')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--quiet',
                'https://github.com/NguyenDuong58/11230945-NguyenBaDuong.git',
                str(REPO_DIR)],check=True)
# Only a published features stage may pin its historic code revision.
# An unpublished feature directory may belong to an interrupted build and
# must not force checkout of an incomplete run's source commit.
experiment_state = json.loads(
    (EXPERIMENT_DIR / 'experiment_manifest.json').read_text(encoding='utf-8')
)
assert experiment_state.get('experiment_id') == EXPERIMENT_ID
published_features = experiment_state.get('stages', {}).get('features')
if published_features is not None:
    assert published_features.get('status') == 'PASSED', (
        'A features stage exists but is not marked PASSED; review its registry state'
    )
    published_manifest = EXPERIMENT_DIR / 'features' / 'part4_manifest.json'
    assert published_manifest.is_file(), 'Published feature manifest missing'
    PART4_COMMIT = json.loads(published_manifest.read_text(encoding='utf-8'))['code_commit']
    assert re.fullmatch(r'[0-9a-f]{40}', PART4_COMMIT)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--quiet', PART4_COMMIT],
                   check=True)
    print('Published feature stage: immutable, read-only verification')
else:
    PART4_COMMIT = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
    ).strip()
    print('Unpublished feature stage: use committed source on main')
assert subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()==PART4_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-r',
                str(REPO_DIR/'requirements_part4.txt')],check=True)
for module in list(sys.modules):
    if module == 'fdic_pipeline' or module.startswith('fdic_pipeline.'):
        del sys.modules[module]
sys.path.insert(0,str(REPO_DIR/'src'))
print('Experiment:',EXPERIMENT_ID)
print('Feature-pipeline Git commit:',PART4_COMMIT)
print('Code checkout:',REPO_DIR)

### 2. Execute the complete regression and integration test suite

The test gate includes Parts 1–4, synthetic Parquet tests of feature and label construction, resolution-type distinctions, chronological boundaries, missingness and train-only preprocessing. Missing dependencies are treated as an invalid environment: **failures, errors, and skipped tests must all equal zero**.

In [ ]:
import importlib.metadata
import subprocess
import xml.etree.ElementTree as ET

for package in ('pandas', 'polars', 'pyspark', 'pyarrow', 'numpy', 'scikit-learn', 'joblib'):
    print(f'{package}: {importlib.metadata.version(package)}')

xml_file = Path('/content/bft65b_part4_tests.xml')
log_file = Path('/content/bft65b_part4_pytest.log')
xml_file.unlink(missing_ok=True)

command = [
    sys.executable, '-m', 'pytest', '-q', '-rs', '--tb=short',
    '--junitxml', str(xml_file), str(REPO_DIR / 'tests'),
]
completed = subprocess.run(
    command, cwd=REPO_DIR, check=False,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
log_file.write_text(completed.stdout, encoding='utf-8')
print(completed.stdout[-40000:])
print('Complete test log:', log_file)

if completed.returncode != 0:
    raise RuntimeError(
        f'Part 4 test gate failed (pytest exit code {completed.returncode}). '
        'Review the failures printed above before running Section 3.'
    )

assert xml_file.is_file(), 'pytest did not create JUnit XML output'
suites = ET.parse(xml_file).getroot().findall('.//testsuite')
assert suites, 'No test results recorded'
TESTS = {
    field: sum(int(suite.get(field, 0)) for suite in suites)
    for field in ('tests', 'failures', 'errors', 'skipped')
}
assert TESTS['failures'] == TESTS['errors'] == TESTS['skipped'] == 0, TESTS
print('Parts 1–4 full test gate: PASSED', TESTS)


### 3. Verify Part 1–3 stage lineage and input schemas

The feature builder reads source panel partitions exclusively through the verified `quality_panel` manifest. The processing-stage validation is retained as experiment provenance; grouped benchmark outputs are not financial predictors. The Failures source and terminal audit are checked separately using their SHA-256 digests.

In [ ]:
from fdic_pipeline.project_flow import get_active,resolve_stage,stage_dir
from fdic_pipeline.feature_pipeline import source_contract,feature_schema,model_schema
ACTIVE=get_active(DRIVE_ROOT)
assert ACTIVE['experiment_id']==EXPERIMENT_ID
for name in ('acquisition','storage','quality_panel','processing'):
    assert resolve_stage(DRIVE_ROOT,name)['status']=='PASSED',name
SNAP=source_contract(DRIVE_ROOT,verify_hashes=True)
SNAP['feature_code_commit']=PART4_COMMIT
assert SNAP['panel_fields']==48
assert len(SNAP['partitions'])==170
assert SNAP['panel_rows']>0
print('Upstream source contract: PASSED')
print('Verified panel observations:',SNAP['panel_rows'])
print('Panel quarters:',len(SNAP['partitions']))
print('Part 2 manifest SHA-256:',SNAP['part2_manifest_sha256'])
print('Part 3 validation SHA-256:',SNAP['processing_verification_sha256'])
print('Predictor feature count:',len(feature_schema().names)-6)

### 4. Lock the feature dictionary, event definition, time split, and leakage policy

Features are computed exclusively from same-quarter or preceding-quarter Financials values. Certificates and reporting dates remain identifiers, not model features. The four-quarter outcome horizon starts at the reporting-quarter end, while availability is checked separately.

**Development:** 1984Q1–2006Q4. **Purge:** 2007Q1–2007Q4. **Test:** 2008Q1–2014Q4. Other periods are retained for source traceability but excluded from modelling. Only reports through 2025Q2 have potentially mature four-quarter labels. These intervals are specified before the test-set model evaluation.

In [ ]:
from fdic_pipeline.feature_policy import (
    FEATURE_NAMES,FEATURE_SPECS,FEATURE_RATIONALES,TARGET_COLUMN,DEVELOPMENT_END,TEST_START,TEST_END,
    LAST_LABEL_QUARTER,STUDY_END,AVAILABILITY_LAG_DAYS,AVAILABILITY_SENSITIVITY,
    EXCLUDED_PREDICTOR_FIELDS,unfit_preprocessor,
)
from fdic_pipeline.feature_pipeline import policy_lock
assert len(FEATURE_NAMES)==21 and len(set(FEATURE_NAMES))==21
assert set(FEATURE_NAMES).isdisjoint(EXCLUDED_PREDICTOR_FIELDS)
POLICY=policy_lock()
assert POLICY['assumption_is_verified_release_date'] is False
assert POLICY['refit_preprocessing_in_every_part5_fold'] is True
print('Target:',TARGET_COLUMN)
print('Development end:',DEVELOPMENT_END,'Test:',TEST_START,'to',TEST_END)
print('Last labelled reporting quarter:',LAST_LABEL_QUARTER)
print('Study end:',STUDY_END)
print('Assumed availability lag:',AVAILABILITY_LAG_DAYS,'days')
print('Development-only lag sensitivity:',AVAILABILITY_SENSITIVITY)
print('Features (group, definition, information set, economic rationale):')
for row in FEATURE_SPECS:
    print(' | '.join((*row, FEATURE_RATIONALES[row[0]])))
print('Unfitted preprocessing policy:',unfit_preprocessor())

### 5. Audit FDIC event resolution types before generating outcomes

`RESTYPE=FAILURE` defines the qualifying outcome. `ASSISTANCE` is not a failure and is conservatively treated as a competing intervention when it occurs during an observation's prediction horizon. Historical failure records without a usable CERT or with uncertain dates remain in the source audit; they are not forcibly linked to unrelated bank records.

The test-calendar count is based on distinct `CERT` values with a dated `FAILURE` event in 2008–2014 and a link to Financials. It is deliberately different from the number of positive test bank-quarter labels, whose four-quarter outcomes may extend past 2014.

In [ ]:
from fdic_pipeline.feature_pipeline import load_events
EVENTS,EVENT_AUDIT=load_events(SNAP)
print('Resolution types in full source:')
for k,v in EVENT_AUDIT['resolution_type_counts'].items():
    print(' ',k,v)
print('2008–2014 FAILURE source rows:',EVENT_AUDIT['test_2008_2014_failure_source_rows'])
print('2008–2014 distinct FAILURE CERT linked to Financials:',
      EVENT_AUDIT['test_2008_2014_distinct_failure_certs_in_panel'])
print('2008–2014 dated source transactions:',EVENT_AUDIT['event_source_rows_by_period']['test_calendar'])
print('Missing failure-reference CERT:',EVENT_AUDIT['source_rows_missing_cert'])
print('CERT with uncertain event evidence:',EVENT_AUDIT['uncertain_certificates_by_reason'])
SOURCE_CERTS=json.loads((Path(SNAP['quality_dir'])/'part2_manifest.json').read_text())['source_certificates']
assert len(EVENTS)==SOURCE_CERTS, 'Event ledger must cover every source CERT'
assert EVENT_AUDIT['test_2008_2014_distinct_failure_certs_in_panel']>=50
print('At least 50 documented linked failures in test calendar: PASSED')

### 6. Construct feature partitions, chronological model samples and training-only preprocessing

The builder processes all 170 verified panel quarters in chronological order. It writes features and eligible labels to Parquet while retaining censored observations and reporting dates in a full feature table. Development and holdout are written to separate files.

The `SimpleImputer` and `RobustScaler` template is fitted only on development feature columns. The fitted artifact does **not** replace chronological-fold fitting in Part 5. Once published, the `features` stage is immutable. An interrupted unpublished stage can be rebuilt without altering Parts 1–3.

In [ ]:
from fdic_pipeline.project_flow import HandoffError
FEATURE_DIR=stage_dir(DRIVE_ROOT,'features')
LOG_FILE=EXPERIMENT_DIR/'run_logs'/'part4_feature_pipeline.log'
try:
    EXISTING=resolve_stage(DRIVE_ROOT,'features')
except HandoffError as exc:
    if 'has not passed verification' not in str(exc):
        raise
    EXISTING=None
if EXISTING is None:
    command=[sys.executable,'-u',str(REPO_DIR/'run_feature_pipeline.py'),'build',
             '--drive-root',str(DRIVE_ROOT),'--code-commit',PART4_COMMIT,
             '--replace-incomplete','--log-file',str(LOG_FILE)]
    subprocess.run(command,cwd=REPO_DIR,check=True)
else:
    print('Published features stage found; verification will be read-only')
assert resolve_stage(DRIVE_ROOT,'features')['status']=='PASSED'
print('Features stage:',FEATURE_DIR)
print('Persistent log:',LOG_FILE)

### 7. Examine sample accounting and label-status evidence

The sample audit distinguishes **dated failure events**, **distinct certificates with a failure event**, **positive bank-quarter labels**, and **eligible modelling observations**. Development and holdout counts must not be substituted for one another. Censored and label-immature reports remain represented in the complete feature table.

In [ ]:
import csv
from IPython.display import display,Markdown

def evidence_table(title,relative,limit=30):
    with (FEATURE_DIR/relative).open(encoding='utf-8',newline='') as fp:
        rows=list(csv.DictReader(fp))
    assert rows,f'Empty evidence file: {relative}'
    fields=list(rows[0])
    print(f'{title}: {len(rows)} rows')
    view=rows[:limit]
    display(Markdown('**'+title+'**'))
    display(Markdown(' | '.join(fields)+'\n'+' | '.join(['---']*len(fields))+'\n'+
        '\n'.join(' | '.join(str(row[k]).replace('|','&#124;') for k in fields) for row in view)))
    if len(rows)>limit:
        print('Showing first',limit,'of',len(rows),'rows; full audit is stored on Drive')

MANIFEST=json.loads((FEATURE_DIR/'part4_manifest.json').read_text(encoding='utf-8'))
print('Full feature table:',MANIFEST['source_rows'])
print('Development eligible rows:',MANIFEST['model_files']['development']['rows'])
print('Test eligible rows:',MANIFEST['model_files']['test_holdout']['rows'])
print('Positive-label distinct CERT:',MANIFEST['distinct_certs_with_positive_labels'])
print('Actual test failure events (by FAILDATE):',
      MANIFEST['source_event_audit']['test_2008_2014_failure_source_rows'])
evidence_table('Chronological sample reconciliation','audits/sample_summary.csv',10)
evidence_table('Outcome and censoring status','audits/label_status.csv',35)
evidence_table('Actual source events by resolution type','audits/failure_events_by_period.csv',5)

### 8. Inspect training-period feature missingness and preprocessing fit

All feature-distribution diagnostics shown here are computed on the **development period**. The saved preprocessing pipeline is fitted using development feature columns alone; the holdout file is not opened by the fitter. A new, unfitted preprocessing pipeline must be created inside every Part 5 chronological validation fold.

In [ ]:
FIT=json.loads((FEATURE_DIR/'audits'/'preprocessing_fit.json').read_text(encoding='utf-8'))
assert FIT['fit_split']=='development'
assert FIT['test_data_used_in_fit'] is False
assert FIT['refit_required_per_validation_fold'] is True
assert FIT['fit_rows']==MANIFEST['model_files']['development']['rows']
assert FIT['feature_names']==list(FEATURE_NAMES)
print('Preprocessor training split:',FIT['fit_split'])
print('Preprocessor fit observations:',FIT['fit_rows'])
print('Input/output feature dimensions:',FIT['feature_count_in'],FIT['feature_count_out'])
print('All-missing development features:',FIT['all_missing_development_features'])
print('Fitted artifact SHA-256:',FIT['joblib_sha256'])
evidence_table('Highest missingness in development','audits/feature_missingness_development.csv',21)
evidence_table('Development-only availability-lag sensitivity',
               'audits/development_availability_sensitivity.csv',3)

### 9. Display saved feature and outcome evidence figures

The three figures show development-period feature missingness, positive development bank-quarter labels by reporting year, and the historical distribution of FDIC `FAILURE` events by event date. The last figure is a descriptive event audit, not a model validation result.

In [ ]:
from IPython.display import Image
for relative in ('figures/development_feature_missingness.png',
                 'figures/development_labels_by_year.png',
                 'figures/qualifying_failures_by_year.png'):
    display(Markdown('**'+Path(relative).stem.replace('_',' ').title()+'**'))
    display(Image(filename=str(FEATURE_DIR/relative),width=820))

### 10. Independently re-verify all feature values, labels and source relationships

The verifier rechecks 170 original panel-file digests, the source failure/audit references, all derived feature files, model-data splits, preprocessing artifact, evidence files and parent-stage hashes. It reconstructs the 21 financial features and the four-quarter outcome of every bank-quarter record, then checks the development and holdout files against eligible rows. This is a read-only audit and does not fit a model.

In [ ]:
from fdic_pipeline.feature_pipeline import verify_project_stage
VERIFIED=verify_project_stage(DRIVE_ROOT)
assert VERIFIED['success'] is True
assert VERIFIED['verified_quarters']==170
assert VERIFIED['feature_rows']==SNAP['panel_rows']
assert VERIFIED['feature_count']==21
assert VERIFIED['assistance_excluded_from_failure_label'] is True
assert VERIFIED['train_only_preprocessing_fit'] is True
assert VERIFIED['historical_publication_times_verified'] is False
assert VERIFIED['read_only_value_verification'] is True
print('Part 4: PASSED')
for field in ('feature_rows','verified_quarters','development_rows','test_holdout_rows',
              'development_positive_bank_quarters','test_positive_bank_quarters',
              'actual_test_failure_source_rows','actual_test_distinct_failure_certs_in_panel',
              'audit_artifacts_verified'):
    print(field+':',VERIFIED[field])
print('Independent source-to-feature-and-label verification: PASSED')

### 11. Establish the locked feature-data contract for Part 5

Part 5 may load `model_data/development.parquet` for model development, selection and cost-sensitive threshold determination. The separate `test_holdout.parquet` must be opened only during the final, single holdout evaluation after a model-selection lock has been written. Development-fold preprocessing must be refitted inside each fold; the Part 4 fitted artifact is a documented full-development transformation, not a substitute for fold-specific fitting.

In [ ]:
FEATURE_STAGE=resolve_stage(DRIVE_ROOT,'features')
assert FEATURE_STAGE['status']=='PASSED'
assert FEATURE_STAGE['validation']['part4_manifest_sha256']==VERIFIED['part4_manifest_sha256']
assert FEATURE_STAGE['parent_verification_sha256']==SNAP['quality_verification_sha256']
assert FEATURE_STAGE['validation']['processing_verification_sha256']==SNAP['processing_verification_sha256']
print('Part 5 training dataset:',FEATURE_DIR/'model_data'/'development.parquet')
print('Part 5 sealed holdout dataset:',FEATURE_DIR/'model_data'/'test_holdout.parquet')
print('Feature policy lock:',FEATURE_DIR/'feature_policy.lock.json')
print('Feature stage validation:',FEATURE_DIR/'part4_validation.json')
print('Feature code revision:',PART4_COMMIT)
print('Handoff state: PASSED')